# Stage 1 Training — Cross-Embodiment Shared Latent Space

Yan et al. (2026) adaptation for dexterous hands.

**Setup checklist:**
- Runtime set to GPU: `Runtime > Change runtime type > T4 GPU`
- Repo synced to `MyDrive/AIST-hand/`
- `dex-urdf/` folder inside `MyDrive/AIST-hand/dex-urdf/`
- `hograspnet_abl11.csv` in `MyDrive/AIST-hand/` (root)
- `shadow_hand_right.yaml` in `MyDrive/AIST-hand/` (root)
- `valid_robot_poses_eigengrasp_dong.npz` in `MyDrive/AIST-hand/` (root) — collision-free Shadow Hand poses

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/AIST-hand')
GITHUB_TOKEN = ''
GITHUB_USER  = 'isapedraza'
REPO_NAME    = 'AIST-hand'

# Training hyperparameters
B               = 50_000    # train batch size
N_STEPS         = 15_000    # training steps
LOG_EVERY       = 50
CKPT_EVERY      = 500
LR_WARMUP       = 500       # unused under Yan-pure constant LR (retained for backward compat)
VAL_EVERY       = 500       # eval val split every N steps
N_EVAL_BATCHES  = 20        # batches per eval pass
B_EVAL          = 5000      # batch size for eval

print(f'Drive root: {DRIVE_ROOT}')
print(f'B={B}, N_STEPS={N_STEPS}, VAL_EVERY={VAL_EVERY}')

## 3. Clone repo from GitHub

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'
BRANCH   = 'run30-pinch-switching'   # Run 30: Run 29 + Xin sigmoid switching pinch (ported from Run 21 paper-sk)

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} fetch origin')

os.system(f'git -C {REPO_DIR} checkout {BRANCH}')
os.system(f'git -C {REPO_DIR} pull origin {BRANCH}')
print(f'Checked out branch: {BRANCH}')

REPO_ROOT = Path(REPO_DIR)
DEX_ROOT  = DRIVE_ROOT / 'dex-urdf'
print(f'Repo: {REPO_ROOT}')
print(f'dex-urdf: {DEX_ROOT}')

## 4. Install dependencies

In [ ]:
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION  = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'torch={TORCH_VERSION}, cuda={CUDA_VERSION}')

!pip install -q torch-geometric
!pip install -q pytorch-kinematics
!pip install -q -e /content/AIST-hand/models/latent-retargeting
!pip install -q -e /content/AIST-hand/human

print('Done.')

## 5. Run Stage 1 training

In [ ]:
import torch

CSV_PATH         = DRIVE_ROOT / 'hograspnet_abl11.csv'
URDF_PATH        = DEX_ROOT  / 'robots/hands/shadow_hand/shadow_hand_right.urdf'
HAND_CONFIG      = DRIVE_ROOT / 'shadow_hand_right.yaml'
ROBOT_YAML       = REPO_ROOT / 'robot/hands/shadow_hand/robot.yaml'
CKPT_PATH        = DRIVE_ROOT / 'checkpoints/stage1_latest.pt'
VALID_POSES_PATH = DRIVE_ROOT / 'valid_robot_poses_eigengrasp_dong.npz'
EXTRA_HUMAN_CSV  = DRIVE_ROOT / 'data/processed/hagrid_dong.csv'

# Run 30 -- Run 29 + Xin sigmoid switching pinch (ported from Run 21 paper-sk).
# Adds 4 helpers fiel to Xin 2025 / Xin code: sigmoid gate s(d), tip_pos suppression
# stilde(d), pinch_rescale l(d), asymmetric anchor target. Per-finger subspace.
# Thumb only in lam_thumb_pos (no double-counting). Pinky no pinch (Xin doesn't define).
RESUME_CKPT = None

# Architecture
SINGLE_LATENT = False
Z_DIM_TOTAL   = 320            # single latent total dim (= 5*64 capacity parity)
Z_DIM         = 64             # used only when SINGLE_LATENT=False (legacy)
SHARED_DIM    = 1024

# Optimizer
LR     = 1e-3
MARGIN = 0.05

# Loss weights
LAMBDA_C     = 10.0
LAMBDA_REC   = 5.0
LAMBDA_LTC   = 1.0
LAMBDA_TMP   = 0.1
LAMBDA_JOINT = 1.0

# Xin S_k intra-weights (same as Run 29)
LAM_TIP_POS   = 1.0
LAM_THUMB_POS = 10.0  # L_thumb_pos (Xin Eq.1) -- 10x other fingertips
LAM_PINCH = 10.0
LAM_TIP_ROT   = 10.0  # L_fingertip_rot (Xin Eq.4)
LAM_PIP_POS   = 1.0   # PIP position (DexMV-style; not in Xin paper)
LAM_DR    = 16.5  # Yan D_R per-finger (filtered by SUBSPACE_LABEL_PREFIX)

# Run 30 NEW: Xin pinch switching. Defaults match Xin paper and Run 21 paper-sk exactly.
ENABLE_PINCH_SWITCHING  = True
PINCH_EPS1_M            = 0.1    # transition threshold (meters)
PINCH_EPS2_M            = 0.01   # in-contact threshold (meters)
PINCH_REF_HAND_LENGTH_M = 0.197  # reference hand length (Xin paper)
PINCH_SIGMOID_W_M       = 10.0   # sigmoid sharpness (1/m)

# Sampling / triplets
N_TRIPLETS = None
EXTRA_HUMAN_RATIO = 0.10
SEED = 21266   # -1 picks a random seed; 21266 = Run 20 reproducible basin

required_paths = [CSV_PATH, HAND_CONFIG, ROBOT_YAML, VALID_POSES_PATH, EXTRA_HUMAN_CSV]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

print(f'CSV         : {CSV_PATH}')
print(f'CKPT        : {CKPT_PATH}')
print(f'VALID_POSES : {VALID_POSES_PATH}')
print(f'ROBOT_YAML  : {ROBOT_YAML}')
print(f'HaGRID extra: {EXTRA_HUMAN_CSV} | ratio={EXTRA_HUMAN_RATIO}')
print(f'RESUME_CKPT : {RESUME_CKPT}')
print(f'Encoder     : {"HumanEncoder_E_h_single (single latent, z_dim_total=" + str(Z_DIM_TOTAL) + ")" if SINGLE_LATENT else "HumanEncoder_E_h (5 subspaces, z_dim=" + str(Z_DIM) + ")"}')
print(f'Xin S_k     : lam_thumb_pos={LAM_THUMB_POS}, lam_tip_pos={LAM_TIP_POS}, lam_pinch={LAM_PINCH}, lam_tip_rot={LAM_TIP_ROT}, lam_pip_pos={LAM_PIP_POS}, lam_dr={LAM_DR}')
print(f'Pinch switch: enable={ENABLE_PINCH_SWITCHING}, eps1_m={PINCH_EPS1_M}, eps2_m={PINCH_EPS2_M}, ref_m={PINCH_REF_HAND_LENGTH_M}, w_m={PINCH_SIGMOID_W_M}')
print(f'L_joint     : lambda_joint={LAMBDA_JOINT}')
print(f'MARGIN      : {MARGIN} | LAMBDA_LTC: {LAMBDA_LTC}')
print(f'Scheduler   : none (Yan-pure constant LR)')
print(f'SEED        : {SEED} ({"random, auto-generated at runtime" if SEED < 0 else "fixed"})')

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "-u",
    str(REPO_ROOT / "models/latent-retargeting/scripts/train_cross_emb.py"),
    "--repo_root",         str(REPO_ROOT),
    "--dex_root",          str(DEX_ROOT),
    "--csv_path",          str(CSV_PATH),
    "--ckpt_path",         str(CKPT_PATH),
    "--hand_config",       str(HAND_CONFIG),
    "--robot_yaml_path",   str(ROBOT_YAML),
    "--valid_poses_path",  str(VALID_POSES_PATH),
    "--extra_human_csv",   str(EXTRA_HUMAN_CSV),
    "--extra_human_ratio", str(EXTRA_HUMAN_RATIO),
    "--b",                 str(B),
    "--n_steps",           str(N_STEPS),
    "--log_every",         str(LOG_EVERY),
    "--ckpt_every",        str(CKPT_EVERY),
    "--lr_warmup",         str(LR_WARMUP),
    "--z_dim",             str(Z_DIM),
    "--z_dim_total",       str(Z_DIM_TOTAL),
    "--shared_dim",        str(SHARED_DIM),
    "--lr",                str(LR),
    "--lambda_c",          str(LAMBDA_C),
    "--lambda_rec",        str(LAMBDA_REC),
    "--lambda_ltc",        str(LAMBDA_LTC),
    "--lambda_tmp",        str(LAMBDA_TMP),
    "--lambda_joint",      str(LAMBDA_JOINT),
    "--lam_tip_pos",       str(LAM_TIP_POS),
    "--lam_thumb_pos",     str(LAM_THUMB_POS),
    "--lam_pinch",         str(LAM_PINCH),
    "--lam_tip_rot",       str(LAM_TIP_ROT),
    "--lam_pip_pos",       str(LAM_PIP_POS),
    "--lam_dr",            str(LAM_DR),
    "--pinch_eps1_m",      str(PINCH_EPS1_M),
    "--pinch_eps2_m",      str(PINCH_EPS2_M),
    "--pinch_ref_hand_length_m", str(PINCH_REF_HAND_LENGTH_M),
    "--pinch_sigmoid_w_m", str(PINCH_SIGMOID_W_M),
    "--margin",            str(MARGIN),
    "--seed",              str(SEED),
    "--val_every",         str(VAL_EVERY),
    "--n_eval_batches",    str(N_EVAL_BATCHES),
    "--b_eval",            str(B_EVAL),
    "--log_metric_stats",
]
if SINGLE_LATENT:
    cmd.append("--single_latent")
if ENABLE_PINCH_SWITCHING:
    cmd.append("--enable_pinch_switching")
if N_TRIPLETS is not None:
    cmd.extend(["--n_triplets", str(N_TRIPLETS)])
if RESUME_CKPT is not None:
    cmd.extend(["--resume_ckpt", str(RESUME_CKPT)])

print('Training command:')
print(' '.join(cmd))

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Training failed with code {proc.returncode}")

In [ ]:
# GPU memory diagnostics -- run after first step to check headroom
import torch
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Allocated : {allocated:.2f} GB")
print(f"Reserved  : {reserved:.2f} GB")
print(f"Total     : {total:.2f} GB")
print(f"Free      : {total - reserved:.2f} GB")